# Exploración Completa del Dataset OASIS3 🧠

## Guía Interactiva para Visualizar y Analizar Datos de Resonancia Magnética

En este notebook aprenderás cómo:
- Cargar y explorar imágenes MRI del dataset OASIS3
- Visualizar volúmenes 3D en diferentes planos
- Analizar estadísticas del dataset
- Extraer y procesar rebanadas de imágenes
- Trabajar con formatos NIfTI y LMDB

## 📦 Instalación de Dependencias

Ejecuta la siguiente celda para instalar las librerías necesarias:

In [ ]:
# Instala el entorno antes de abrir el notebook:
# python -m pip install -e '.[toolkit]'
print("✓ Usando el entorno activo del proyecto")

## 🔧 Importar Librerías

In [ ]:
import os
import re
import sys
from pathlib import Path
import csv
import json
import pickle
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import nibabel as nib
import lmdb

# Configurar matplotlib
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✓ Todas las librerías importadas correctamente")

## 📍 Configuración de Rutas

Define dónde están tus datos OASIS3:

In [ ]:
# Rutas portables; se pueden sobrescribir con variables de entorno
search_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in search_roots if (p / 'src' / 't5_ssl').is_dir()), Path.cwd())
DATA_ROOT = Path(os.environ.get("OASIS3_DATA_ROOT", REPO_ROOT / "data" / "dementia"))
CSV_PATH = Path(os.environ.get("OASIS3_CSV", DATA_ROOT / "mni-dataset.csv"))
OASIS3_ROOT = Path(os.environ.get("OASIS3_IMAGE_ROOT", DATA_ROOT / "oasis3"))
LMDB_PATH = Path(os.environ.get("OASIS3_LMDB", DATA_ROOT / "mni-dataset.lmdb"))

# Verificar que las rutas existen
print("Verificando rutas...")
print(f"📄 CSV:        {CSV_PATH.exists()} - {CSV_PATH}")
print(f"📂 OASIS3:     {OASIS3_ROOT.exists()} - {OASIS3_ROOT}")
print(f"🗄️  LMDB:       {LMDB_PATH.exists()} - {LMDB_PATH}")

## 🛠️ Funciones Auxiliares

In [ ]:
def load_and_normalize_volume(image_path):
    """Cargar archivo NIfTI y normalizar a [0, 1]."""
    img = nib.load(image_path)
    volume = np.asarray(nib.as_closest_canonical(img).get_fdata(dtype=np.float32))
    
    # Normalización robusta usando percentiles
    foreground = volume[volume != 0]
    if foreground.size > 0:
        low, high = np.percentile(foreground, [1, 99])
        volume = np.clip((volume - low) / max(high - low, 1e-6), 0, 1)
    
    return volume, img.affine


def plot_ortho_planes(volume, title="MRI Volume"):
    """Visualizar volumen en los 3 planos principales."""
    x, y, z = volume.shape
    mid_x, mid_y, mid_z = x // 2, y // 2, z // 2
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    
    # Axial
    axes[0].imshow(volume[:, :, mid_z], cmap='gray')
    axes[0].set_title(f'Axial (Z={mid_z})', fontweight='bold')
    axes[0].grid(True, alpha=0.2)
    
    # Sagital
    axes[1].imshow(volume[mid_x, :, :], cmap='gray', origin='lower')
    axes[1].set_title(f'Sagital (X={mid_x})', fontweight='bold')
    axes[1].grid(True, alpha=0.2)
    
    # Coronal
    axes[2].imshow(volume[:, mid_y, :], cmap='gray', origin='lower')
    axes[2].set_title(f'Coronal (Y={mid_y})', fontweight='bold')
    axes[2].grid(True, alpha=0.2)
    
    plt.tight_layout()
    plt.show()


def print_volume_stats(volume, image_path=None):
    """Imprimir estadísticas del volumen."""
    print(f"Shape:          {volume.shape}")
    print(f"Dtype:          {volume.dtype}")
    print(f"Range:          [{volume.min():.4f}, {volume.max():.4f}]")
    print(f"Mean:           {volume.mean():.4f}")
    print(f"Std:            {volume.std():.4f}")
    print(f"Non-zero voxels: {np.count_nonzero(volume):,} / {volume.size:,}")


print("✓ Funciones auxiliares definidas")

## 1️⃣ Carga de Datos desde CSV

Comencemos cargando y explorando el archivo CSV clínico:

In [ ]:
# Leer CSV
records = []
with open(CSV_PATH, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row.get('study_name', '').strip().upper() == 'OASIS3':
            records.append(row)

print(f"✓ Total de registros OASIS3: {len(records):,}")
print(f"\nPrimeros 3 registros:")
for i, record in enumerate(records[:3]):
    print(f"\n{i+1}. Subject: {record.get('subject', 'N/A'):20s} | Session: {record.get('original_code', 'N/A'):25s} | Group: {record.get('group', 'N/A')}")

## 2️⃣ Análisis Estadístico del Dataset

In [ ]:
# Estadísticas básicas
def subject_for(record):
    explicit = record.get('subject', '').strip()
    if explicit:
        return explicit
    code = record.get('original_code', '').strip()
    match = re.match(r'(sub-OAS\d+)', code)
    return match.group(1) if match else code

unique_subjects = len({subject_for(r) for r in records})
group_counts = defaultdict(int)
records_per_subject = defaultdict(int)

for record in records:
    subject = subject_for(record)
    group = record.get('group', '').strip().upper()
    
    group_counts[group] += 1
    records_per_subject[subject] += 1

print("📊 ESTADÍSTICAS DEL DATASET")
print("="*50)
print(f"Total de registros: {len(records):,}")
print(f"Sujetos únicos: {unique_subjects:,}")
print(f"\nDistribución por grupo:")
for group, count in sorted(group_counts.items(), key=lambda x: x[1], reverse=True):
    pct = 100 * count / len(records)
    print(f"  {group:15s}: {count:6,} ({pct:5.1f}%)")

records_values = list(records_per_subject.values())
print(f"\nRegistros por sujeto:")
print(f"  Media: {np.mean(records_values):.2f}")
print(f"  Mediana: {np.median(records_values):.0f}")
print(f"  Rango: {min(records_values)}-{max(records_values)}")

## 3️⃣ Visualizar Distribuciones

In [ ]:
# Gráficos de distribución
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de grupos
groups = sorted(group_counts.items(), key=lambda x: x[1], reverse=True)
group_names = [g[0] for g in groups]
group_values = [g[1] for g in groups]
colors = plt.cm.Set3(np.linspace(0, 1, len(group_names)))

axes[0].bar(group_names, group_values, color=colors, edgecolor='black', linewidth=1.5)
axes[0].set_title('Distribución de Grupos Clínicos', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Número de Registros', fontsize=11)
axes[0].grid(axis='y', alpha=0.3)

# Histograma de registros por sujeto
axes[1].hist(records_values, bins=20, color='skyblue', edgecolor='black', alpha=0.7)
axes[1].axvline(np.mean(records_values), color='red', linestyle='--', linewidth=2, label=f'Media: {np.mean(records_values):.1f}')
axes[1].set_title('Registros por Sujeto', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Número de Registros', fontsize=11)
axes[1].set_ylabel('Número de Sujetos', fontsize=11)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4️⃣ Carga de un Volumen NIfTI Individual

In [ ]:
# Buscar un archivo NIfTI existente
nifti_files = list(OASIS3_ROOT.glob("*_T1w.nii.gz"))

if nifti_files:
    sample_file = nifti_files[0]
    print(f"Archivo de ejemplo: {sample_file.name}")
    
    # Cargar volumen
    volume, affine = load_and_normalize_volume(sample_file)
    volume_title = f'Volumen NIfTI: {sample_file.name}'
    
    print(f"\n📊 Información del volumen:")
    print_volume_stats(volume)
else:
    print("⚠️  No se encontraron archivos NIfTI; usando una muestra LMDB")
    if not (LMDB_PATH / 'data.mdb').exists():
        raise FileNotFoundError('No se encontró ni NIfTI ni LMDB')
    env = lmdb.open(str(LMDB_PATH), readonly=True, lock=False, readahead=False)
    with env.begin() as txn:
        key, value = next(iter(txn.cursor()))
        payload = pickle.loads(value)
    env.close()
    volume = payload['image'].astype(np.float32)
    foreground = volume[volume != 0]
    low, high = np.percentile(foreground, [1, 99])
    volume = np.clip((volume - low) / max(high - low, 1e-6), 0, 1)
    affine = None
    volume_title = f'Volumen LMDB: {key.decode("ascii")}'
    print_volume_stats(volume)

## 5️⃣ Visualizar el Volumen en 3 Planos

In [ ]:
plot_ortho_planes(volume, title=volume_title)

## 6️⃣ Extracción de Rebanadas (Slices)

In [ ]:
# Extraer rebanadas axiales equidistantes
z_size = volume.shape[2]
n_slices = 9

# Seleccionar posiciones equidistantes (excluyendo bordes)
z_positions = np.linspace(int(z_size * 0.1), int(z_size * 0.9), n_slices, dtype=int)
print(f"Rebanadas seleccionadas en Z: {z_positions}")

# Visualizar galería de rebanadas
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle('Galería de Rebanadas Axiales (Centro del Volumen)', fontsize=14, fontweight='bold')

for idx, z_pos in enumerate(z_positions):
    ax = axes[idx // 3, idx % 3]
    slice_data = volume[:, :, z_pos]
    
    im = ax.imshow(slice_data, cmap='gray')
    ax.set_title(f'Z = {z_pos}', fontweight='bold')
    ax.grid(True, alpha=0.2)
    
    # Colorbar
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 7️⃣ Análisis de Intensidades en una Rebanada

In [ ]:
# Analizar una rebanada específica
z_center = z_size // 2
slice_center = volume[:, :, z_center]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Imagen
im = axes[0].imshow(slice_center, cmap='gray')
axes[0].set_title(f'Rebanada Axial Central (Z={z_center})', fontweight='bold')
plt.colorbar(im, ax=axes[0])

# Histograma de intensidades
foreground = slice_center[slice_center > 0.01]  # Excluir fondo
axes[1].hist(foreground.flatten(), bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[1].set_title('Distribución de Intensidades', fontweight='bold')
axes[1].set_xlabel('Intensidad', fontsize=11)
axes[1].set_ylabel('Frecuencia', fontsize=11)
axes[1].grid(alpha=0.3)

print(f"Estadísticas de la rebanada Z={z_center}:")
print(f"  Min: {slice_center.min():.4f}")
print(f"  Max: {slice_center.max():.4f}")
print(f"  Mean: {foreground.mean():.4f}")
print(f"  Std: {foreground.std():.4f}")
print(f"  Voxels no-cero: {np.count_nonzero(foreground):,}")

plt.tight_layout()
plt.show()

## 8️⃣ Explorar LMDB (Almacén Optimizado)

Si tienes acceso a la base de datos LMDB, puedes cargar volúmenes más rápidamente:

In [ ]:
# Verificar si LMDB existe
lmdb_data_file = LMDB_PATH / "data.mdb"

if lmdb_data_file.exists():
    print(f"✓ LMDB encontrado en {LMDB_PATH}")
    
    # Abrir LMDB (solo lectura)
    env = lmdb.open(str(LMDB_PATH), readonly=True, lock=False, readahead=False, max_readers=64)
    
    # Obtener información básica
    stat = env.stat()
    print(f"\n📦 Información de LMDB:")
    print(f"  Número de claves: {stat['entries']:,}")
    print(f"  Tamaño máximo: {env.info()['map_size']:,} bytes")
    
    # Mostrar las primeras claves
    print(f"\nPrimeras 5 claves (índices):")
    with env.begin() as txn:
        cursor = txn.cursor()
        for i, (key, value) in enumerate(cursor):
            if i >= 5:
                break
            print(f"  {i+1}. {key.decode('ascii')}")
    
    env.close()
else:
    print("⚠️  LMDB no encontrado en", LMDB_PATH)

## 9️⃣ Cargar Volumen desde LMDB

In [ ]:
if lmdb_data_file.exists():
    # Abrir LMDB y cargar un volumen
    env = lmdb.open(str(LMDB_PATH), readonly=True, lock=False, readahead=False, max_readers=64)
    
    # Obtener la primera clave
    with env.begin() as txn:
        cursor = txn.cursor()
        key, value = next(iter(cursor))
        key_str = key.decode('ascii')
        
        # Deserializar el valor (pickle)
        payload = pickle.loads(value)
        
        print(f"✓ Volumen cargado desde LMDB: {key_str}")
        print(f"\nContenido:")
        for k, v in payload.items():
            if k != 'image':
                print(f"  {k}: {v}")
            else:
                print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
        
        # Normalizar la imagen
        lmdb_volume = payload['image'].astype(np.float32)
        foreground = lmdb_volume[lmdb_volume != 0]
        if foreground.size > 0:
            low, high = np.percentile(foreground, [1, 99])
            lmdb_volume = np.clip((lmdb_volume - low) / max(high - low, 1e-6), 0, 1)
        
        plot_ortho_planes(lmdb_volume, title=f"Volumen LMDB: {key_str}")
    
    env.close()

## 🔟 Ejercicios Prácticos

### Ejercicio 1: Comparar dos volúmenes
Carga dos volúmenes diferentes y compáralos.

In [ ]:
# TODO: Implementar comparación de dos volúmenes
# Ayuda: Puedes cargar múltiples archivos NIfTI y visualizarlos lado a lado

### Ejercicio 2: Identificar sujetos con Alzheimer
Filtra y cuenta los sujetos diagnosticados con Alzheimer.

In [ ]:
# Contar sujetos AD únicos
ad_subjects = set()
for record in records:
    if record.get('group', '').strip().upper() == 'AD':
        ad_subjects.add(subject_for(record))

print(f"Total de sujetos con Alzheimer (AD): {len(ad_subjects):,}")
print(f"Ejemplos: {list(ad_subjects)[:5]}")

### Ejercicio 3: Crear tu propia galería
Visualiza 15 rebanadas del volumen en una composición personalizada.

In [ ]:
# Crear galería de 15 rebanadas (5x3)
n_slices_full = 15
z_positions_full = np.linspace(0, z_size - 1, n_slices_full, dtype=int)

fig, axes = plt.subplots(5, 3, figsize=(12, 20))
fig.suptitle('Galería Completa del Volumen (Todas las Rebanadas)', fontsize=14, fontweight='bold')

for idx, z_pos in enumerate(z_positions_full):
    ax = axes[idx // 3, idx % 3]
    slice_data = volume[:, :, z_pos]
    
    z_fraction = z_pos / z_size
    if z_fraction < 0.3:
        anatomy = "(Inferior)"
    elif z_fraction < 0.7:
        anatomy = "(Central)"
    else:
        anatomy = "(Superior)"
    
    ax.imshow(slice_data, cmap='gray')
    ax.set_title(f'Z = {z_pos} {anatomy}', fontsize=9)
    ax.grid(True, alpha=0.2)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

## 📚 Conceptos Clave para Recordar

### Planos Ortogonales
- **Axial (Z):** Vista desde arriba, hacia abajo (transversal)
- **Sagital (X):** Vista lateral, de izquierda a derecha
- **Coronal (Y):** Vista frontal, de anterior a posterior

### Normalización
Las imágenes se normalizan usando percentiles robustos (1% y 99%) para mapear valores a [0, 1].
Esto maneja variaciones de intensidad entre máquinas y pacientes.

### Formato de Datos
- **NIfTI (.nii.gz):** Archivos comprimidos, fáciles de compartir pero lentos de leer
- **LMDB:** Base de datos indexada, rápida para entrenamiento de modelos

### Metadatos
El CSV contiene información clínica: grupo (AD/HC), sesión, etc.

## 🎓 Conclusiones

Ahora puedes:
✅ Cargar y visualizar imágenes MRI
✅ Entender la estructura del dataset OASIS3
✅ Extraer y analizar rebanadas
✅ Trabajar con NIfTI y LMDB

## 🚀 Próximos Pasos

1. Explora más volúmenes del dataset
2. Busca patrones en sujetos AD vs HC
3. Implementa segmentación de tejidos
4. Entrena un modelo de deep learning

¡Diviértete explorando el cerebro! 🧠